In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/Users/ishaanjain/DL PROJECT/data_raw/diabetes_dataset__2019_bitmesra.csv')
df.rename(columns={'Pregancies': 'Pregnancies'}, inplace=True)
df.columns = df.columns.str.strip()
df

,Age,Gender,Family_Diabetes,highBP,PhysicallyActive,BMI,Smoking,Alcohol,Sleep,SoundSleep,RegularMedicine,JunkFood,Stress,BPLevel,Pregnancies,Pdiabetes,UriationFreq,Diabetic
0,50-59,Male,no,yes,one hr or more,39.0,no,no,8,6,no,occasionally,sometimes,high,0.0,0,not much,no
1,50-59,Male,no,yes,less than half an hr,28.0,no,no,8,6,yes,very often,sometimes,normal,0.0,0,not much,no
2,40-49,Male,no,no,one hr or more,24.0,no,no,6,6,no,occasionally,sometimes,normal,0.0,0,not much,no
3,50-59,Male,no,no,one hr or more,23.0,no,no,8,6,no,occasionally,sometimes,normal,0.0,0,not much,no
4,40-49,Male,no,no,less than half an hr,27.0,no,no,8,8,no,occasionally,sometimes,normal,0.0,0,not much,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
947,less than 40,Male,yes,no,more than half an hr,25.0,no,no,8,6,no,often,sometimes,normal,0.0,0,not much,yes
948,60 or older,Male,yes,yes,more than half an hr,27.0,no,no,6,5,yes,occasionally,sometimes,high,0.0,0,quite often,yes
949,60 or older,Male,no,yes,none,23.0,no,no,6,5,yes,occasionally,sometimes,high,0.0,0,not much,no
950,60 or older,Male,no,yes,less than half an hr,27.0,no,yes,6,5,yes,occasionally,very often,high,0.0,0,not much,no


In [102]:
df.isnull().sum()

Age                  0
Gender               0
Family_Diabetes      0
highBP               0
PhysicallyActive     0
BMI                  4
Smoking              0
Alcohol              0
Sleep                0
SoundSleep           0
RegularMedicine      0
JunkFood             0
Stress               0
BPLevel              0
Pregnancies         42
Pdiabetes            1
UriationFreq         0
Diabetic             1
dtype: int64

In [104]:
# 2. Fill missing numerical values
# Median is used for BMI to avoid influence of outliers
df['BMI'] = df['BMI'].fillna(df['BMI'].median())
# Assume missing pregnancy data means 0
df['Pregnancies'] = df['Pregnancies'].fillna(0)

# 3. Drop rows where the target (Diabetic) is missing
df.dropna(subset=['Diabetic','Pdiabetes'], inplace=True)

In [106]:
# 1. Convert all text to lowercase and strip hidden leading/trailing spaces
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip().str.lower()

# 2. Fix specific data-entry errors found in the BIT Mesra file
df['RegularMedicine'] = df['RegularMedicine'].replace('o', 'no')
df['Pdiabetes'] = df['Pdiabetes'].replace('0', 'no')
df['BPLevel'] = df['BPLevel'].replace('normal ', 'normal') # Standardizing BP categories


In [108]:
# Define logical scales (Higher Number = Higher Clinical Risk/Intensity)
age_map = {'less than 40': 0, '40-49': 1, '50-59': 2, '60 or older': 3}
active_map = {'none': 0, 'less than half an hr': 1, 'more than half an hr': 2, 'one hr or more': 3}
junk_map = {'occasionally': 1, 'often': 2, 'very often': 3, 'always': 4}
stress_map = {'not at all': 0, 'sometimes': 1, 'very often': 2, 'always': 3}
bp_map = {'low': 0, 'normal': 1, 'high': 2}

# Apply mappings to the columns
df['Age'] = df['Age'].map(age_map)
df['PhysicallyActive'] = df['PhysicallyActive'].map(active_map)
df['JunkFood'] = df['JunkFood'].map(junk_map)
df['Stress'] = df['Stress'].map(stress_map)
df['BPLevel'] = df['BPLevel'].map(bp_map)


In [110]:
df

,Age,Gender,Family_Diabetes,highBP,PhysicallyActive,BMI,Smoking,Alcohol,Sleep,SoundSleep,RegularMedicine,JunkFood,Stress,BPLevel,Pregnancies,Pdiabetes,UriationFreq,Diabetic
0,2,male,no,yes,3,39.0,no,no,8,6,no,1,1,2,0.0,no,not much,no
1,2,male,no,yes,1,28.0,no,no,8,6,yes,3,1,1,0.0,no,not much,no
2,1,male,no,no,3,24.0,no,no,6,6,no,1,1,1,0.0,no,not much,no
3,2,male,no,no,3,23.0,no,no,8,6,no,1,1,1,0.0,no,not much,no
4,1,male,no,no,1,27.0,no,no,8,8,no,1,1,1,0.0,no,not much,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
947,0,male,yes,no,2,25.0,no,no,8,6,no,2,1,1,0.0,no,not much,yes
948,3,male,yes,yes,2,27.0,no,no,6,5,yes,1,1,2,0.0,no,quite often,yes
949,3,male,no,yes,0,23.0,no,no,6,5,yes,1,1,2,0.0,no,not much,no
950,3,male,no,yes,1,27.0,no,yes,6,5,yes,1,2,2,0.0,no,not much,no


In [112]:
# 1. Map all binary categories (Yes/No, Gender, Urination)
binary_map = {
    'yes': 1, 'no': 0, 
    'male': 1, 'female': 0,
    'quite often': 1, 'not much': 0
}

binary_cols = ['Gender', 'Family_Diabetes', 'highBP', 'Smoking', 'Alcohol', 
               'RegularMedicine', 'Pdiabetes', 'UriationFreq', 'Diabetic']

for col in binary_cols:
    df[col] = df[col].map(binary_map)

# 2. Final verification: Check if any non-numeric data remains
print(f"Non-numeric columns remaining: {df.select_dtypes(exclude=[np.number]).columns.tolist()}")

Non-numeric columns remaining: []


In [114]:
df

,Age,Gender,Family_Diabetes,highBP,PhysicallyActive,BMI,Smoking,Alcohol,Sleep,SoundSleep,RegularMedicine,JunkFood,Stress,BPLevel,Pregnancies,Pdiabetes,UriationFreq,Diabetic
0,2,1,0,1,3,39.0,0,0,8,6,0,1,1,2,0.0,0,0,0
1,2,1,0,1,1,28.0,0,0,8,6,1,3,1,1,0.0,0,0,0
2,1,1,0,0,3,24.0,0,0,6,6,0,1,1,1,0.0,0,0,0
3,2,1,0,0,3,23.0,0,0,8,6,0,1,1,1,0.0,0,0,0
4,1,1,0,0,1,27.0,0,0,8,8,0,1,1,1,0.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
947,0,1,1,0,2,25.0,0,0,8,6,0,2,1,1,0.0,0,0,1
948,3,1,1,1,2,27.0,0,0,6,5,1,1,1,2,0.0,0,1,1
949,3,1,0,1,0,23.0,0,0,6,5,1,1,1,2,0.0,0,0,0
950,3,1,0,1,1,27.0,0,1,6,5,1,1,2,2,0.0,0,0,0


In [ ]:
#Save as csv
df.to_csv('diabetes_cleaned_dataset.csv', index=False)
print(" Dataset saved as 'diabetes_cleaned_logic.csv'.")
df.head()

 Dataset saved as 'diabetes_cleaned_logic.csv'.


,Age,Gender,Family_Diabetes,highBP,PhysicallyActive,BMI,Smoking,Alcohol,Sleep,SoundSleep,RegularMedicine,JunkFood,Stress,BPLevel,Pregnancies,Pdiabetes,UriationFreq,Diabetic
0,2,1,0,1,3,39.0,0,0,8,6,0,1,1,2,0.0,0,0,0
1,2,1,0,1,1,28.0,0,0,8,6,1,3,1,1,0.0,0,0,0
2,1,1,0,0,3,24.0,0,0,6,6,0,1,1,1,0.0,0,0,0
3,2,1,0,0,3,23.0,0,0,8,6,0,1,1,1,0.0,0,0,0
4,1,1,0,0,1,27.0,0,0,8,8,0,1,1,1,0.0,0,0,0
